# Re-run with the fixed Summarizer (Spanish facts) - Kaggle
Qwen2.5-3B writes, Qwen2.5-7B verifies, on the 16 frozen articles. Saves per-fact evidence for a NEW hand-labelling round (a held-out set for the rules and the judge).

**Before running:** Session options → **Accelerator: GPU T4 x2**, **Internet: On**. About 15-20 min.

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('newsroom-agents'):
    !git clone https://github.com/Juanangara/newsroom-agents.git
%cd /kaggle/working/newsroom-agents
!git pull -q
!git log --oneline | head -1
!pip install -q pydantic feedparser trafilatura googlenewsdecoder bitsandbytes accelerate

In [ ]:
import sys, json, torch
sys.path.insert(0, '/kaggle/working/newsroom-agents/src')
print('GPUs:', torch.cuda.device_count())
from newsroom.benchmark import run_benchmark

SNAPSHOTS = ['data/ia_colombia.json', 'data/reforma_pensional.json']
res = run_benchmark(['Qwen/Qwen2.5-3B-Instruct'], SNAPSHOTS,
                    verifier_id='Qwen/Qwen2.5-7B-Instruct', tag='diag_es', out_dir='/kaggle/working/out')

## Sanity check: are the facts in Spanish now?

In [ ]:
from newsroom.agents import looks_english
ev = res[0]['evidence']
print('facts:', len(ev), '| written in English:', sum(looks_english(e['fact']) for e in ev))
print('grounding:', res[0]['grounding_rate'], '| drop reasons:', res[0]['drop_reasons'])
for e in ev[:5]: print('-', e['fact'])

## Build the labelling sheet

In [ ]:
!python scripts/make_label_sheet.py /kaggle/working/out/benchmark_diag_es.json /kaggle/working/out/to_label_es.csv
!ls -la /kaggle/working/out

Download from the **Output** tab: `out/benchmark_diag_es.json` and `out/to_label_es.csv`.